In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re
import plotly.express as px

In [3]:
cases=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases/cases_cleaned_v3.csv')

In [4]:
cases['case_cause'].value_counts().to_frame

<bound method Series.to_frame of case_cause
35:271 Patent Infringement                             49865
28:1338 Patent Infringement                            12142
15:1126 Patent Infringement                             7991
35:145 Patent Infringement                              7943
35:0271 Patent Infringement                             2313
                                                       ...  
33:1365 Environmental Matters                              1
15:0044 Trademark Infringement                             1
28:0158  Notice of Appeal re Bankruptcy Matter (BA         1
28:2255 Motion to Vacate / Correct Illegal Sentence        1
42:2651  Medical Care Recovery                             1
Name: count, Length: 451, dtype: int64>

In [5]:
legal_code_mapping = {
    "35:271": "35:271 Patent Infringement",
    "28:2201": "28:2201 Declaratory Judgment",
    "28:1338": "28:1338 Federal IP Jurisdiction",
    "28:1331": "28:1331 Federal Question",
    "15:1125": "15:1125 Trademark Infringement (Lanham Act)",
    "35:183": "35:183 Right to Compensation",
    "35:145": "35:145 Civil Action to Obtain Patent",
    "35:292": "35:292 False Patent Marking",
    "15:1051": "15:1051 Application for Registration; Verification",
    "15:1126": "15:1126 International Conventions",
    "35:1": "35:1 Establishment",
    "35:281": "35:281: Remedy for Infringement of Patent",
    "28:1441": "28:1441: Removal of Civil Actions",
    "17:101": "17:101: Definitions",
    "15:1114": "15:1114: Remedies and Procedure for Infringement of Registered Marks",
    "35:154": "35:154: Contents and Term of Patent; Provisional Rights",
    "28:1332": "28:1332: Diversity of Citizenship; Amount in Controversy; Costs",
    "35:101": "35:101: Inventions Patentable",
    "15:1121": "15:1121 Trademark — Jurisdiction of Federal Courts",
    "28:1391": "28:1391 Venue Generally",
    "17:501": "17:501 Copyright Infringement",
    "35:100": "35:100 Conditions for Patentability; Novelty",
    "35:146": "35:146 Civil Action in Case of Interference",
    "31:3729": "31:3729 False Claims Act",
    "35:256": "35:256 Correction of Named Inventor",
    "28:1446": "28:1446 Procedure for Removal of Civil Actions",
    "35:292": "35:292 False Patent Marking",
    "5:702":  "5:702 APA — Right of Review",
    "15:1127": "15:1127 Trademark — Construction and Definitions",
    "18:1341": "18:1341 Mail Fraud",
    "28:1442": "28:1442 Federal Officer or Agency — Removal",
    "5:706":  "5:706 APA — Scope of Review",
    "42:1983": "42:1983 Civil Action for Deprivation of Rights",
    "15:1":   "15:1 Sherman Antitrust Act",
    "28:1440": "28:1440 Removal — Insurance Regulations",
    "35:102": "35:102 Conditions for Patentability; Prior Art",
    "5:551":  "5:551 APA — Definitions",
    "35:283": "35:283 Patent — Injunction",
    "35:282": "35:282 Patent — Presumption of Validity; Defenses",
    "45:51":  "45:51 Federal Employers' Liability Act (FELA)",
    "28:2254": "28:2254 Habeas Corpus — State Custody",
    "28:1361": "28:1361 Mandamus Actions",
    "7:2321": "7:2321 Plant Variety Protection Act",
    "18:1836": "18:1836 Defend Trade Secrets Act",
    "28:1330": "28:1330 Actions Against Foreign States",
    "15:2":   "15:2 Sherman Act — Monopolization",
    "5:701":  "5:701 APA — Application",
    "28:1337": "28:1337 Commerce and Antitrust Jurisdiction",
    "18:1961": "18:1961 RICO — Definitions",
    "15:15":  "15:15 Clayton Act — Suits by Persons Injured",
    "35:284": "35:284 Patent — Damages",
    "28:2241": "28:2241 Habeas Corpus — Power to Grant Writ",
    "35:289": "35:289 Design Patent Infringement Damages",
    "9:1":    "9:1 Federal Arbitration Act — Definitions",
    "9:10":   "9:10 FAA — Vacation of Awards",
    "9:9":    "9:9 FAA — Confirmation of Arbitration Awards",
    "17:504": "17:504 Copyright — Damages and Profits",
    "28:1343": "28:1343 Civil Rights Jurisdiction",
    "28:1346": "28:1346 United States as Defendant",
    "18:241": "18:241 Conspiracy Against Rights",
    "15:1692": "15:1692 Fair Debt Collection Practices Act",
    "28:1407": "28:1407 Multidistrict Litigation",
    "28:1651": "28:1651 All Writs Act",
    "18:1030": "18:1030 Computer Fraud and Abuse Act",
    "28:2255": "28:2255 Federal Prisoner — Motion to Vacate Sentence",
    "18:1964": "18:1964 RICO — Civil Remedies",
    "15:1601": "15:1601 Truth in Lending Act",
    "45:151": "45:151 Railway Labor Act",
    "33:1365": "33:1365 Clean Water Act — Citizen Suits",
    "31:3731": "31:3731 False Claims Act — Civil Investigative Demand",
    "15:1116": "15:1116 Trademark — Injunctive Relief",
    "28:1631": "28:1631 Transfer to Cure Want of Jurisdiction",
    "15:1071": "15:1071 Trademark — Appeal to Courts",
    "28:1400": "28:1400 Patent and Copyright Venue",
    "28:1782": "28:1782 Assistance to Foreign Tribunals",
    "5:704":  "5:704 APA — Actions Reviewable",
    "15:1681": "15:1681 Fair Credit Reporting Act",
    "15:45":  "15:45 FTC Act — Unfair Methods of Competition",
    "15:52":  "15:52 FTC Act — False Advertisements",
    "28:2409": "28:2409 Quiet Title Actions Against United States",
    "29:201": "29:201 Fair Labor Standards Act",
    "42:12101": "42:12101 Americans with Disabilities Act",
    "18:1962": "18:1962 RICO — Prohibited Activities",
    "18:2511": "18:2511 Electronic Communications Privacy Act",
    "42:4332": "42:4332 NEPA — Environmental Impact Statements",
    "18:981": "18:981 Civil Asset Forfeiture",
    "29:401": "29:401 Labor-Management Reporting and Disclosure Act",
    "12:22":  "12:22 National Bank Act — General Powers",
    "29:1001": "29:1001 ERISA — Congressional Findings",
    "29:1109": "29:1109 ERISA — Liability for Breach of Fiduciary Duty",
    "35:41":  "35:41 Patent Fees",
    "28:1334": "28:1334 Bankruptcy Jurisdiction",
    "35:172": "35:172 Right of Priority for Design Patents",
    "18:1030": "18:1030 Computer Fraud and Abuse Act",
    "35:32":  "35:32 Suspension or Exclusion from Practice",
    "42:2000": "42:2000 Civil Rights Act — Equal Employment",
    "28:1345": "28:1345 United States as Plaintiff",
    "15:1":   "15:1 Sherman Act — Restraint of Trade",
    "12:635": "12:635 Export-Import Bank Act",
    "47:227": "47:227 Restrictions on Telephone Solicitation",
    "28:158": "28:158 Bankruptcy Appellate Panels",
    "21:348": "21:348 Food, Drug, and Cosmetic Act — Food Additives",
    "30:281": "30:281 Mineral Leasing Act",
    "18:1836": "18:1836 Defend Trade Secrets Act",
    "35:311": "35:311 Inter Partes Reexamination",
    "26:7428": "26:7428 Tax Court — Declaratory Judgments",
    "38:1780": "38:1780 Veterans — Educational Assistance",
    "0:0": "Unknown",
    "88:8888": "Unknown",
    "99:9999": "Unknown",
    "15:44": "15:44 Definitions", 
    "28:2271": "28:2271 Rule-making Power Generally",
    "15:78": "15:78 Registration and Regulation of Brokers and Dealers",
    "42:405": "42:405 Evidence, Procedure, and Certification for Payments",
    "42:205": "42:205 Appointment and Tenure of Office of Surgeon General; Reversion in Rank",
    "28:7422": "28:7422 Cause of Action",
    "2:431": "2:431 Transferred",
    "11:1": "11:1 Bankruptcy",
    "42:2651": "42:2651 Recovery by United States",
    "35:1331": "35:331 Inter Partes Review",
    "38:271": "35:271 Patent Infringement",
    "17:43": "17:43 Copyright — Repealed/Historical",
    "35:291": "35:291 Invalidity of Patent",
    "18:4208": "18:4208 Parole Eligibility",
    "28:1983": "28:1983 Civil Rights",
    "28:157": "28:157 Judges of the Court of International Trade",
    "29:754": "29:754 Labor Management Relations Act",
    "5:304": "5:304 Congressional Review Act",
    "32:102": "35:102 Conditions for Patentability; Prior Art",
    "28:1444": "28:1444 Petition for Removal - Foreclosure",
    "35:217": "35:271 Patent Infringement",
    "38:2011": "35:271) Patent Infringement",
    "28:138": "28:138 Terms Abolished",
    "15:281": "15:281 Structural Failures",
    "28:1443": "28:1443 Civil Rights Cases",
    "28:1201": "Unknown",
    "31:3545": "31:3545 Civil Action to Recover Money",
    "8:1105": "8:1105Liaison with internal security officers; data exchange",
    "12:1819": "12:1819 Default of Promissory Note",
    "38:775": "38:775 Recovery of Servicemen's Group Life Insurance",
    "2:437": "02:437 Federal Election Commission",
    "08:1451": "08:1451 Revoke Citizenship",
    "24:271": "24:271 National Cemeteries (Repealed 1973)",
    "45:371": "45:371 Railroad Unemployment Insurance Act", 
    "15:101": "Unknown",
    "8:287": "Unknown",
    "8:1451": "8:1451 Revoke Citizenship",
    "5:70": "5:70 Government Organization and Employees (Repealed/Renumbered)",
    "7:6": "7:6 Federal Commodity Exchange Regulation",
    "15:25": "15:25 Clayton Act",
    "15:271": "Unknown",
    "18:271": "Unknown",
    "15:1501": "15:1501 Establishment of Department of Commerce",
    "36:1": "36:1 Patriotic and National Observances", 
    
    
    
    
    
    

}

# --- 2. Normalization function (fixes formatting + leading zeros)
def normalize_statute(text):
    if pd.isna(text):
        return "Unknown"
    
    if isinstance(text, float) or isinstance(text, int):
        return "Unknown"
    
    match = re.search(r'(\d+)\s*:\s*0*(\d+)', str(text))
    
    if not match:
        return "Unknown"

    title = int(match.group(1))
    section = int(match.group(2))
    key = f"{title}:{section}"
    return legal_code_mapping.get(key, key)

# --- 3. Apply to dataframe
cases["case_cause_uscode"] = cases["case_cause"].apply(normalize_statute)
from IPython.display import display, HTML

vc = cases['case_cause_uscode'].value_counts()

display(HTML(f"""
<div style="
    height: 400px;
    overflow-y: scroll;
    border: 1px solid #ccc;
    padding: 10px;
">
{vc.to_frame().to_html()}
</div>
"""))

,count
case_cause_uscode,
35:271 Patent Infringement,52243
28:1338 Federal IP Jurisdiction,12879
35:145 Civil Action to Obtain Patent,8962
15:1126 International Conventions,8103
Unknown,3802
28:2201 Declaratory Judgment,3229
28:1331 Federal Question,1818
35:183 Right to Compensation,1016
15:1125 Trademark Infringement (Lanham Act),825


In [6]:
print(f"Total rows: {len(cases)}")
print(f"Total non-null in case_cause_uscode: {cases['case_cause_uscode'].notna().sum()}")
print(f"Total null in case_cause_uscode: {cases['case_cause_uscode'].isna().sum()}")

Total rows: 96881
Total non-null in case_cause_uscode: 96881
Total null in case_cause_uscode: 0


In [8]:

threshold = 200
vc = cases['case_cause_uscode'].value_counts()

main = vc[vc >= threshold]
other_count = vc[vc < threshold].sum()


main = vc[vc >= threshold]
other_count = vc[vc < threshold].sum()

chart_data = pd.concat([main, pd.Series({"Other": other_count})])
chart_data = chart_data.sort_values()

# high contrast color sequence
colors = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4",
    "#469990", "#dcbeff", "#9A6324", "#BF1616", "#800000",
    "#aaffc3", "#808000", "#470505", "#000075", "#a9a9a9"
]

fig = px.bar(
    chart_data,
    orientation='h',
    labels={"index": "US Code", "value": "Number of Cases"},
    color=chart_data.index,
    color_discrete_sequence=colors
)

fig.update_layout(
    height=600,
    xaxis_title="Number of Cases",
    margin=dict(l=300),
    yaxis_title="",
    showlegend=False,
    title={
        'text': "<b><i>Patent Litigation Cases by Cause of Action 1972-2020</i></b>",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    }
)

fig.show()
